# LangGraph - Продвинутые техники

## Что изучим:

1. **Checkpoints** - сохранение и восстановление состояния
2. **Human-in-the-loop** - человек проверяет перед действием
3. **Streaming** - потоковый вывод токенов

### Зачем это нужно?

**Production проблемы:**
- Агент упал посередине работы - все потеряно
- Агент хочет удалить файл - нет подтверждения
- Долгий ответ - пользователь ждет

**LangGraph решения:**
- Checkpoints - сохраняем состояние, можно продолжить
- Human-in-the-loop - пауза для подтверждения
- Streaming - токены летят сразу

In [5]:
# Установка
!pip install -q langgraph langchain langchain-openai python-dotenv

In [6]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")
print(f"API Key: {api_key[:10]}..." if api_key else "API Key не найден!")

API Key: sk-***...


---

## 1. Checkpoints - Сохранение состояния

### Проблема:

Агент работает 5 минут, упал на 99-м документе - все потеряно.

### Решение - Checkpointer:

Каждый шаг сохраняется. Упал? Продолжаем с последнего checkpoint!

### Java аналогия:

Checkpoint - это как Saga Pattern или Event Sourcing:
- Event Sourcing: история всех шагов, восстановление из событий
- Activiti/Camunda: каждый шаг workflow сохраняется в БД

In [7]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
import operator

# Состояние
class ConversationState(TypedDict):
    messages: Annotated[list, operator.add]
    step: int

# Узел обработки
def process_node(state: ConversationState) -> ConversationState:
    step = state.get("step", 0) + 1
    print(f"\nШаг {step}")
    
    last_msg = state["messages"][-1] if state["messages"] else ""
    response = f"Ответ на шаге {step}: получил '{last_msg}'"
    
    return {
        "messages": [response],
        "step": step
    }

# Граф
workflow = StateGraph(ConversationState)
workflow.add_node("process", process_node)
workflow.set_entry_point("process")
workflow.add_edge("process", END)

# КЛЮЧЕВОЙ МОМЕНТ: добавляем checkpointer
checkpointer = MemorySaver()
app = workflow.compile(checkpointer=checkpointer)

print("Граф с checkpoints создан!")

Граф с checkpoints создан!


In [8]:
# Первое сообщение в thread
config = {"configurable": {"thread_id": "user-123"}}

result1 = app.invoke(
    {"messages": ["Привет!"], "step": 0},
    config=config
)
print(f"Результат 1: {result1}")


Шаг 1
Результат 1: {'messages': ['Привет!', "Ответ на шаге 1: получил 'Привет!'"], 'step': 1}


In [9]:
# Второе сообщение в ТОМ ЖЕ thread - состояние сохранилось!
result2 = app.invoke(
    {"messages": ["Как дела?"], "step": 0},  # step будет проигнорирован
    config=config  # тот же thread_id
)
print(f"Результат 2: {result2}")
print(f"\nЗаметь: step увеличился автоматически!")


Шаг 1
Результат 2: {'messages': ['Привет!', "Ответ на шаге 1: получил 'Привет!'", 'Как дела?', "Ответ на шаге 1: получил 'Как дела?'"], 'step': 1}

Заметь: step увеличился автоматически!


In [10]:
# Можно получить историю состояний
history = list(app.get_state_history(config))
print(f"Количество checkpoints: {len(history)}")

for i, state in enumerate(history):
    print(f"\n--- Checkpoint {i} ---")
    print(f"Step: {state.values.get('step')}")
    print(f"Messages: {state.values.get('messages', [])[-2:]}")

Количество checkpoints: 6

--- Checkpoint 0 ---
Step: 1
Messages: ['Как дела?', "Ответ на шаге 1: получил 'Как дела?'"]

--- Checkpoint 1 ---
Step: 0
Messages: ["Ответ на шаге 1: получил 'Привет!'", 'Как дела?']

--- Checkpoint 2 ---
Step: 1
Messages: ['Привет!', "Ответ на шаге 1: получил 'Привет!'"]

--- Checkpoint 3 ---
Step: 1
Messages: ['Привет!', "Ответ на шаге 1: получил 'Привет!'"]

--- Checkpoint 4 ---
Step: 0
Messages: ['Привет!']

--- Checkpoint 5 ---
Step: None
Messages: []


### Что дает Checkpoint:

1. **Память между вызовами** - thread_id связывает сессии
2. **История** - можно посмотреть все шаги
3. **Восстановление** - продолжить после ошибки
4. **Time travel** - вернуться к предыдущему состоянию

### Типы Checkpointer:

| Тип | Где хранит | Когда использовать |
|-----|------------|-------------------|
| MemorySaver | RAM | Разработка, тесты |
| SqliteSaver | SQLite файл | Локальный прод |
| PostgresSaver | PostgreSQL | Production |

---

## 2. Human-in-the-loop - Проверка человеком

### Проблема:

Агент хочет отправить email, удалить файлы, сделать покупку - без подтверждения опасно!

### Решение - interrupt_before:

Агент останавливается перед опасным узлом. Человек проверяет и решает: продолжить или отменить.

### Java аналогия:

Human-in-the-loop - это как Approval Workflow в Camunda/Activiti, или как в банках когда большие переводы требуют одобрения менеджера.

In [12]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

!pip install ipywidgets

# Опасный инструмент!
@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Отправляет email. ОПАСНО - требует подтверждения!"""
    print(f"\nОТПРАВКА EMAIL:")
    print(f"   To: {to}")
    print(f"   Subject: {subject}")
    print(f"   Body: {body[:50]}...")
    return f"Email отправлен на {to}"

@tool
def search_contacts(query: str) -> str:
    """Безопасный поиск контактов."""
    return f"Найден контакт: {query}@example.com"

tools = [send_email, search_contacts]
print("Инструменты созданы!")

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 914.9/914.9 kB 8.3 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ----------------------- ---------------- 1.3/2.2 MB 6.7 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 6.9 MB/s  0:00:00

   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]

Инструменты созданы!


In [13]:
# Состояние агента
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

# LLM
llm = ChatOpenAI(
    base_url="https://api.deepseek.com",
    api_key=api_key,
    model="deepseek-chat",
    temperature=0
)
llm_with_tools = llm.bind_tools(tools)

# Узел агента
def agent_node(state: AgentState) -> AgentState:
    print("\nАгент думает...")
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# Узел инструментов
def tools_node(state: AgentState) -> AgentState:
    print("\nВыполняю инструменты...")
    last_message = state["messages"][-1]
    tools_dict = {t.name: t for t in tools}
    
    tool_messages = []
    for tool_call in last_message.tool_calls:
        result = tools_dict[tool_call["name"]].invoke(tool_call["args"])
        tool_messages.append(
            ToolMessage(content=str(result), tool_call_id=tool_call["id"])
        )
    
    return {"messages": tool_messages}

# Решение: продолжить или завершить
def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    return "tools" if last.tool_calls else "end"

print("Узлы созданы!")

Узлы созданы!


In [ ]:
# Создаем граф
workflow = StateGraph(AgentState)

workflow.add_node("agent", agent_node)
workflow.add_node("tools", tools_node)

workflow.set_entry_point("agent")

workflow.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", "end": END}
)
workflow.add_edge("tools", "agent")

# КЛЮЧЕВОЙ МОМЕНТ: interrupt_before
checkpointer = MemorySaver()
app = workflow.compile(
    checkpointer=checkpointer,
    interrupt_before=["tools"]  # Пауза ПЕРЕД выполнением инструментов!
)

print("Граф с Human-in-the-loop создан!")

In [ ]:
# Запрос на отправку email
config = {"configurable": {"thread_id": "email-thread-1"}}

result = app.invoke(
    {"messages": [HumanMessage(content="Отправь email Ивану с темой 'Встреча' и текстом 'Привет! Давай встретимся завтра.'")]},
    config=config
)

print("\n" + "="*50)
print("АГЕНТ ОСТАНОВЛЕН ПЕРЕД ОТПРАВКОЙ!")
print("="*50)

In [ ]:
# Смотрим что агент хочет сделать
state = app.get_state(config)
last_message = state.values["messages"][-1]

print("Агент хочет выполнить:")
for tc in last_message.tool_calls:
    print(f"\n   Tool: {tc['name']}")
    print(f"   Args: {tc['args']}")

print("\n" + "="*50)
print("Одобряете? (в реальности - UI с кнопками)")
print("="*50)

In [ ]:
# ОДОБРИТЬ - продолжить выполнение
print("Одобрено! Продолжаем...")

# Просто вызываем invoke с None - продолжает с checkpoint
result = app.invoke(None, config=config)

print("\n" + "="*50)
print("РЕЗУЛЬТАТ:")
print(result["messages"][-1].content)

### Паттерны Human-in-the-loop:

| Паттерн | Описание | Когда использовать |
|---------|----------|-------------------|
| interrupt_before | Пауза ДО узла | Подтверждение действия |
| interrupt_after | Пауза ПОСЛЕ узла | Проверка результата |
| update_state | Изменить состояние | Отмена, корректировка |

---

## 3. Streaming - Потоковый вывод

### Проблема:

Долгий ответ - пользователь ждет 30 секунд и думает что сломалось.

### Решение - stream:

Токены летят сразу, пользователь видит процесс генерации.

In [ ]:
# Простой граф для демонстрации streaming
from langchain_core.messages import HumanMessage

class StreamState(TypedDict):
    messages: Annotated[list, operator.add]

def stream_agent(state: StreamState) -> StreamState:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

workflow = StateGraph(StreamState)
workflow.add_node("agent", stream_agent)
workflow.set_entry_point("agent")
workflow.add_edge("agent", END)

stream_app = workflow.compile()

print("Граф для streaming создан!")

In [ ]:
# Обычный invoke - ждем весь ответ
import time

print("Обычный invoke (ждем весь ответ):")
start = time.time()

result = stream_app.invoke({
    "messages": [HumanMessage(content="Объясни что такое машинное обучение в 3 предложениях")]
})

print(f"\nВремя: {time.time() - start:.2f}s")
print(f"\nОтвет: {result['messages'][-1].content}")

In [ ]:
# Streaming - видим процесс
print("Streaming (видим события):")
print("="*50)

for event in stream_app.stream({
    "messages": [HumanMessage(content="Объясни что такое машинное обучение в 3 предложениях")]
}):
    print(f"\nEvent: {list(event.keys())}")
    for node_name, output in event.items():
        if "messages" in output:
            print(f"   Content: {output['messages'][-1].content[:100]}...")

In [ ]:
# stream_mode="messages" - потоковые токены
print("Потоковые токены:")
print("="*50)

for msg, metadata in stream_app.stream(
    {"messages": [HumanMessage(content="Напиши хайку про Python")]},
    stream_mode="messages"
):
    if hasattr(msg, 'content') and msg.content:
        print(msg.content, end="", flush=True)

print("\n" + "="*50)

### Режимы streaming:

| Режим | Что возвращает | Когда использовать |
|-------|----------------|-------------------|
| values | Полное состояние после узла | Отладка |
| updates | Только изменения | Эффективность |
| messages | Токены по одному | Chat UI |

---

## Итоги

### Что изучили:

| Техника | Зачем | Как |
|---------|-------|-----|
| **Checkpoints** | Память между вызовами | checkpointer=MemorySaver() |
| **Human-in-the-loop** | Подтверждение действий | interrupt_before=["node"] |
| **Streaming** | UX при долгих ответах | app.stream(...) |

### Java аналогии:

- Checkpoints = Event Sourcing / Saga Pattern
- Human-in-the-loop = Approval Workflow (Camunda)
- Streaming = Reactive Streams / Project Reactor

### Следующие шаги:

1. **Multi-agent** - несколько агентов работают вместе
2. **Subgraphs** - модульная архитектура
3. **Production** - деплой в реальное приложение